In [ ]:
!pip install transformers sentencepiece pandas tqdm torch -q

In [ ]:
pip install torch==2.5.1 torchvision torchaudio --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 773.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Extract Noun phrases with adjectives

CPU- UPPERCASE WITH SENTNECE SAMPLING

In [ ]:
#!/usr/bin/env python3
"""
Extract all unique noun phrases (including multi-word nouns)
from a CSV file (column: 'text') using SpaCy.
Strips leading stop words from noun phrases before storage.
Only keeps noun phrases where majority of words start with capital letter.
Outputs a deduplicated CSV with phrase text, POS type ('NOUN_PHRASE'),
count, and 10 sample unique sentences containing each phrase.
"""
import os
import pandas as pd
from tqdm import tqdm
import spacy
from collections import defaultdict

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_CSV   = "/media/owusus/Godstestimo/NLP-Projects/GhanaNamedEntities/news.csv"
TEXT_COLUMN = "text"          # column that holds the sentences
OUTPUT_FILE = "/media/owusus/Godstestimo/NLP-Projects/GhanaNamedEntities/named_entities_news.csv"
BATCH_SIZE  = 50000
SAVE_EVERY  = 50000
MAX_SAMPLES = 100              # Number of sample sentences per noun phrase

# ------------------------------------------------------------------
# Load SpaCy model
print("🔤 Loading SpaCy model ('en_core_web_sm')...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# Get SpaCy's stop words
stop_words = nlp.Defaults.stop_words
print(f"📋 Loaded {len(stop_words)} stop words from SpaCy")

# ------------------------------------------------------------------
# Load sentences from CSV
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_CSV}")
df_in = pd.read_csv(INPUT_CSV)

if TEXT_COLUMN not in df_in.columns:
    raise KeyError(f"Column '{TEXT_COLUMN}' not found in {INPUT_CSV}")

sentences = df_in[TEXT_COLUMN].dropna().astype(str).str.strip()
sentences = sentences[sentences != ""].tolist()
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Resume logic
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    noun_dict = {}
    for _, row in processed_df.iterrows():
        if pd.notna(row["phrase"]) and isinstance(row["phrase"], str):
            key = row["phrase"].lower()
            # Parse sample sentences from string representation
            samples = []
            if "sample_sentences" in row and pd.notna(row["sample_sentences"]):
                samples_str = str(row["sample_sentences"])
                # Handle list-like string format
                if samples_str.startswith('[') and samples_str.endswith(']'):
                    try:
                        samples = eval(samples_str)
                    except:
                        samples = [samples_str]
                else:
                    samples = [samples_str]

            noun_dict[key] = {
                "phrase": row["phrase"],
                "count": int(row["count"]) if "count" in row and pd.notna(row["count"]) else 0,
                "samples": list(set(samples))[:MAX_SAMPLES] if samples else []
            }
    print(f"🔁 Resuming from {len(noun_dict)} extracted noun phrases.")
else:
    noun_dict = {}

# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def strip_leading_stopwords(phrase, stop_words):
    """Remove stop words from the beginning of a phrase."""
    words = phrase.split()
    while words and words[0].lower() in stop_words:
        words.pop(0)
    return " ".join(words)

def majority_capitalized(phrase):
    """
    Check if majority of words in phrase start with capital letter.
    Returns True if >50% of words are capitalized.
    """
    words = phrase.split()
    if not words:
        return False

    capitalized_count = sum(1 for word in words if word and word[0].isupper())
    return capitalized_count > len(words) / 2

def normalize_phrase(phrase):
    """Create a normalized version for matching (lowercase, no extra spaces)."""
    return " ".join(phrase.lower().split())

# ------------------------------------------------------------------
# Process sentences in batches
# ------------------------------------------------------------------
print("🚀 Starting noun phrase extraction...")
for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[i:i + BATCH_SIZE]
    docs = list(nlp.pipe(batch, disable=["ner"]))

    # Keep track of original sentences for this batch
    batch_sentences = batch

    for sent_idx, doc in enumerate(docs):
        original_sentence = batch_sentences[sent_idx]

        for chunk in doc.noun_chunks:
            phrase = chunk.text.strip()
            if phrase:
                # Strip leading stop words
                phrase = strip_leading_stopwords(phrase, stop_words)

                # Only store if there's still content after stripping
                # AND if majority of words start with capital letter
                if phrase and majority_capitalized(phrase):
                    key = phrase.lower()

                    if key not in noun_dict:
                        noun_dict[key] = {
                            "phrase": phrase,
                            "count": 0,
                            "samples": []
                        }

                    entry = noun_dict[key]
                    entry["count"] += 1

                    # Prefer the version with more capitalized words (better casing)
                    phrase_caps = sum(1 for w in phrase.split() if w and w[0].isupper())
                    orig_caps = sum(1 for w in entry["phrase"].split() if w and w[0].isupper())
                    if phrase_caps > orig_caps:
                        entry["phrase"] = phrase

                    # Add sample sentence if unique and under limit
                    clean_sentence = original_sentence.strip()
                    if clean_sentence not in entry["samples"] and len(entry["samples"]) < MAX_SAMPLES:
                        entry["samples"].append(clean_sentence)

    # Periodic checkpoint
    if (i + BATCH_SIZE) % SAVE_EVERY < BATCH_SIZE:
        temp_df = pd.DataFrame([
            {
                "phrase": data["phrase"],
                "pos": "NOUN_PHRASE",
                "count": data["count"],
                "sample_sentences": data["samples"]
            }
            for data in noun_dict.values()
        ]).sort_values("count", ascending=False)
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Checkpoint saved ({len(noun_dict)} unique noun phrases).")

# ------------------------------------------------------------------
# Final save
# ------------------------------------------------------------------
final_df = pd.DataFrame([
    {
        "phrase": data["phrase"],
        "pos": "NOUN_PHRASE",
        "count": data["count"],
        "sample_sentences": data["samples"]
    }
    for data in noun_dict.values()
]).sort_values("count", ascending=False).reset_index(drop=True)

final_df.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Extracted {len(final_df)} unique noun phrases.")
print("🔝 Top 10 most frequent noun phrases:")
print(final_df.head(10)[["phrase", "count"]].to_string(index=False))
print(f"\n📁 Output saved to: {OUTPUT_FILE}")

🔤 Loading SpaCy model ('en_core_web_sm')...
📋 Loaded 326 stop words from SpaCy
📂 Loading sentences from: /media/owusus/Godstestimo/NLP-Projects/GhanaNamedEntities/news.csv
📄 Loaded 3860295 sentences
🚀 Starting noun phrase extraction...


Processing:   1%|▍                             | 1/78 [01:32<1:58:30, 92.34s/it]

💾 Checkpoint saved (25501 unique noun phrases).


Processing:   3%|▋                            | 2/78 [03:30<2:15:58, 107.34s/it]

💾 Checkpoint saved (44739 unique noun phrases).


...

💾 Checkpoint saved (567628 unique noun phrases).


Processing:  51%|█████████████▎            | 40/78 [2:11:49<2:29:40, 236.33s/it]

💾 Checkpoint saved (580111 unique noun phrases).


Processing:  53%|█████████████▋            | 41/78 [2:15:07<2:18:39, 224.84s/it]

💾 Checkpoint saved (592911 unique noun phrases).


CPU - UPPERCASE ONLY

In [2]:
#!/usr/bin/env python3
"""
Extract all unique noun phrases (including multi-word nouns)
from a CSV file (column: 'text') using SpaCy.
Strips leading stop words from noun phrases before storage.
Only keeps noun phrases where majority of words start with capital letter.
Outputs a deduplicated CSV with phrase text, POS type ('NOUN_PHRASE'), and count.
"""
import os
import pandas as pd
from tqdm import tqdm
import spacy

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_CSV   = "/content/drive/MyDrive/Collab/GhanaNME/news.csv"
TEXT_COLUMN = "text"          # column that holds the sentences
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaNouns/named_entities_news.csv"
BATCH_SIZE  = 10000
SAVE_EVERY  = 50000

# ------------------------------------------------------------------
# Load SpaCy model
print("🔤 Loading SpaCy model ('en_core_web_sm')...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# Get SpaCy's stop words
stop_words = nlp.Defaults.stop_words
print(f"📋 Loaded {len(stop_words)} stop words from SpaCy")

# ------------------------------------------------------------------
# Load sentences from CSV
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_CSV}")
df_in = pd.read_csv(INPUT_CSV)

if TEXT_COLUMN not in df_in.columns:
    raise KeyError(f"Column '{TEXT_COLUMN}' not found in {INPUT_CSV}")

sentences = df_in[TEXT_COLUMN].dropna().astype(str).str.strip()
sentences = sentences[sentences != ""].tolist()
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Resume logic
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    noun_dict = {(row["phrase"].lower()): (row["phrase"], row["count"])
                 for _, row in processed_df.iterrows()
                 if pd.notna(row["phrase"]) and isinstance(row["phrase"], str)}
    print(f"🔁 Resuming from {len(noun_dict)} extracted noun phrases.")
else:
    noun_dict = {}

# ------------------------------------------------------------------
# Helper function to strip leading stop words
# ------------------------------------------------------------------
def strip_leading_stopwords(phrase, stop_words):
    """Remove stop words from the beginning of a phrase."""
    words = phrase.split()
    while words and words[0].lower() in stop_words:
        words.pop(0)
    return " ".join(words)

def majority_capitalized(phrase):
    """
    Check if majority of words in phrase start with capital letter.
    Returns True if >50% of words are capitalized.
    """
    words = phrase.split()
    if not words:
        return False

    capitalized_count = sum(1 for word in words if word and word[0].isupper())
    return capitalized_count > len(words) / 2

# ------------------------------------------------------------------
# Process sentences in batches
# ------------------------------------------------------------------
print("🚀 Starting noun phrase extraction...")
for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[i:i + BATCH_SIZE]
    docs = list(nlp.pipe(batch, disable=["ner"]))

    for doc in docs:
        for chunk in doc.noun_chunks:
            phrase = chunk.text.strip()
            if phrase:
                # Strip leading stop words
                phrase = strip_leading_stopwords(phrase, stop_words)

                # Only store if there's still content after stripping
                # AND if majority of words start with capital letter
                if phrase and majority_capitalized(phrase):
                    key = phrase.lower()
                    if key in noun_dict:
                        orig, count = noun_dict[key]
                        # Prefer the version with more capitalized words (better casing)
                        phrase_caps = sum(1 for w in phrase.split() if w and w[0].isupper())
                        orig_caps = sum(1 for w in orig.split() if w and w[0].isupper())
                        if phrase_caps > orig_caps:
                            orig = phrase
                        noun_dict[key] = (orig, count + 1)
                    else:
                        noun_dict[key] = (phrase, 1)

    # Periodic checkpoint
    if (i + BATCH_SIZE) % SAVE_EVERY < BATCH_SIZE:
        temp_df = pd.DataFrame([
            {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
            for (_, (orig, c)) in noun_dict.items()
        ]).sort_values("count", ascending=False)
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Checkpoint saved ({len(noun_dict)} unique noun phrases).")

# ------------------------------------------------------------------
# Final save
# ------------------------------------------------------------------
final_df = (pd.DataFrame([
                {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
                for (_, (orig, c)) in noun_dict.items()
            ])
            .sort_values("count", ascending=False)
            .reset_index(drop=True))

final_df.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Extracted {len(final_df)} unique noun phrases.")
print("🔝 Top 10 most frequent noun phrases:")
print(final_df.head(10)[["phrase", "count"]].to_string(index=False))

🔤 Loading SpaCy model ('en_core_web_sm')...
📋 Loaded 326 stop words from SpaCy
📂 Loading sentences from: /content/drive/MyDrive/Collab/GhanaNME/news.csv
📄 Loaded 3860295 sentences
🚀 Starting noun phrase extraction...


Processing:   1%|          | 3/387 [01:24<2:59:56, 28.12s/it]


KeyboardInterrupt: 

CPU - LOWERCASE ONLY

In [ ]:
#!/usr/bin/env python3
"""
Extract all unique noun phrases (including multi-word nouns)
from a CSV file (column: 'text') using SpaCy.
Strips leading stop words from noun phrases before storage.
Only keeps noun phrases that are all lowercase.
Outputs a deduplicated CSV with phrase text, POS type ('NOUN_PHRASE'), and count.
"""
import os
import pandas as pd
from tqdm import tqdm
import spacy

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_CSV   = "/content/drive/MyDrive/Collab/GhanaNouns/clean_sentences_speech.csv"
TEXT_COLUMN = "text"          # column that holds the sentences
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaNouns/noun_phrases_speech.csv"
BATCH_SIZE  = 10000
SAVE_EVERY  = 50000

# ------------------------------------------------------------------
# Load SpaCy model
print("🔤 Loading SpaCy model ('en_core_web_sm')...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# Get SpaCy's stop words
stop_words = nlp.Defaults.stop_words
print(f"📋 Loaded {len(stop_words)} stop words from SpaCy")

# ------------------------------------------------------------------
# Load sentences from CSV
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_CSV}")
df_in = pd.read_csv(INPUT_CSV)

if TEXT_COLUMN not in df_in.columns:
    raise KeyError(f"Column '{TEXT_COLUMN}' not found in {INPUT_CSV}")

sentences = df_in[TEXT_COLUMN].dropna().astype(str).str.strip()
sentences = sentences[sentences != ""].tolist()
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Resume logic
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    noun_dict = {(row["phrase"].lower()): (row["phrase"], row["count"])
                 for _, row in processed_df.iterrows()
                 if pd.notna(row["phrase"]) and isinstance(row["phrase"], str)}
    print(f"🔁 Resuming from {len(noun_dict)} extracted noun phrases.")
else:
    noun_dict = {}

# ------------------------------------------------------------------
# Helper function to strip leading stop words
# ------------------------------------------------------------------
def strip_leading_stopwords(phrase, stop_words):
    """Remove stop words from the beginning of a phrase."""
    words = phrase.split()
    while words and words[0].lower() in stop_words:
        words.pop(0)
    return " ".join(words)

# ------------------------------------------------------------------
# Process sentences in batches
# ------------------------------------------------------------------
print("🚀 Starting noun phrase extraction...")
for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[i:i + BATCH_SIZE]
    docs = list(nlp.pipe(batch, disable=["ner"]))

    for doc in docs:
        for chunk in doc.noun_chunks:
            phrase = chunk.text.strip()
            if phrase:
                # Strip leading stop words
                phrase = strip_leading_stopwords(phrase, stop_words)

                # Only store if there's still content after stripping
                # AND if the phrase is all lowercase
                if phrase and phrase == phrase.lower():
                    key = phrase.lower()
                    if key in noun_dict:
                        orig, count = noun_dict[key]
                        # Prefer the version with uppercase letters (but now we only keep lowercase)
                        if phrase != key and orig == key:
                            # New phrase has uppercase, old one doesn't
                            orig = phrase
                        noun_dict[key] = (orig, count + 1)
                    else:
                        noun_dict[key] = (phrase, 1)

    # Periodic checkpoint
    if (i + BATCH_SIZE) % SAVE_EVERY < BATCH_SIZE:
        temp_df = pd.DataFrame([
            {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
            for (_, (orig, c)) in noun_dict.items()
        ]).sort_values("count", ascending=False)
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Checkpoint saved ({len(noun_dict)} unique noun phrases).")

# ------------------------------------------------------------------
# Final save
# ------------------------------------------------------------------
final_df = (pd.DataFrame([
                {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
                for (_, (orig, c)) in noun_dict.items()
            ])
            .sort_values("count", ascending=False)
            .reset_index(drop=True))

final_df.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Extracted {len(final_df)} unique noun phrases.")
print("🔝 Top 10 most frequent noun phrases:")
print(final_df.head(10)[["phrase", "count"]].to_string(index=False))

🔤 Loading SpaCy model ('en_core_web_sm')...
📋 Loaded 326 stop words from SpaCy
📂 Loading sentences from: /content/drive/MyDrive/Collab/GhanaNouns/clean_sentences_speech.csv
📄 Loaded 446970 sentences
🚀 Starting noun phrase extraction...


Processing:  11%|█         | 5/45 [03:33<28:40, 43.01s/it]

💾 Checkpoint saved (60698 unique noun phrases).


Processing:  22%|██▏       | 10/45 [07:01<24:40, 42.31s/it]

💾 Checkpoint saved (92140 unique noun phrases).


...

Processing:  89%|████████▉ | 40/45 [1:10:02<11:28, 137.63s/it]

💾 Checkpoint saved (586716 unique noun phrases).


Processing: 100%|██████████| 45/45 [1:21:44<00:00, 108.99s/it]

💾 Checkpoint saved (640831 unique noun phrases).



✅ Done! Extracted 640831 unique noun phrases.
🔝 Top 10 most frequent noun phrases:
    phrase  count
    people 145181
      time  67022
government  66410
    things  61456
   country  60789
 president  46600
       lot  45384
      fact  40239
     money  34107
     point  33804


GPU - LOWERCASE ONLY

In [ ]:
#!/usr/bin/env python3
"""
Extract all unique noun phrases (including multi-word nouns)
from a CSV file (column: 'text') using SpaCy.
Strips leading stop words from noun phrases before storage.
Only keeps noun phrases that are all lowercase.
Outputs a deduplicated CSV with phrase text, POS type ('NOUN_PHRASE'), and count.
"""
import os
import pandas as pd
from tqdm import tqdm
import spacy

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_CSV   = "/content/drive/MyDrive/Collab/GhanaNouns/deduped_sentences.csv"
TEXT_COLUMN = "text"          # column that holds the sentences
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaNouns/noun_phrases_research.csv"
BATCH_SIZE  = 10000
SAVE_EVERY  = 50000

# ------------------------------------------------------------------
# Load SpaCy model with GPU if available
# ------------------------------------------------------------------
print("🔤 Loading SpaCy model ('en_core_web_sm')...")

# Try to use GPU, fall back to CPU if not available
try:
    spacy.require_gpu()
    print("✅ Using GPU acceleration")
except Exception as e:
    print(f"⚠️  GPU not available, using CPU: {e}")

nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# Get SpaCy's stop words
stop_words = nlp.Defaults.stop_words
print(f"📋 Loaded {len(stop_words)} stop words from SpaCy")

# ------------------------------------------------------------------
# Load sentences from CSV
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_CSV}")
df_in = pd.read_csv(INPUT_CSV)

if TEXT_COLUMN not in df_in.columns:
    raise KeyError(f"Column '{TEXT_COLUMN}' not found in {INPUT_CSV}")

sentences = df_in[TEXT_COLUMN].dropna().astype(str).str.strip()
sentences = sentences[sentences != ""].tolist()
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Resume logic
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    noun_dict = {(row["phrase"].lower()): (row["phrase"], row["count"])
                 for _, row in processed_df.iterrows()
                 if pd.notna(row["phrase"]) and isinstance(row["phrase"], str)}
    print(f"🔁 Resuming from {len(noun_dict)} extracted noun phrases.")
else:
    noun_dict = {}

# ------------------------------------------------------------------
# Helper function to strip leading stop words
# ------------------------------------------------------------------
def strip_leading_stopwords(phrase, stop_words):
    """Remove stop words from the beginning of a phrase."""
    words = phrase.split()
    while words and words[0].lower() in stop_words:
        words.pop(0)
    return " ".join(words)

# ------------------------------------------------------------------
# Process sentences in batches
# ------------------------------------------------------------------
print("🚀 Starting noun phrase extraction...")
for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[i:i + BATCH_SIZE]

    # nlp.pipe() will use GPU automatically if spacy.require_gpu() succeeded
    docs = list(nlp.pipe(batch, disable=["ner"]))

    for doc in docs:
        for chunk in doc.noun_chunks:
            phrase = chunk.text.strip()
            if phrase:
                # Strip leading stop words
                phrase = strip_leading_stopwords(phrase, stop_words)

                # Only store if there's still content after stripping
                # AND if the phrase is all lowercase
                if phrase and phrase == phrase.lower():
                    key = phrase.lower()
                    if key in noun_dict:
                        orig, count = noun_dict[key]
                        noun_dict[key] = (orig, count + 1)
                    else:
                        noun_dict[key] = (phrase, 1)

    # Periodic checkpoint
    if (i + BATCH_SIZE) % SAVE_EVERY < BATCH_SIZE:
        temp_df = pd.DataFrame([
            {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
            for (_, (orig, c)) in noun_dict.items()
        ]).sort_values("count", ascending=False)
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Checkpoint saved ({len(noun_dict)} unique noun phrases).")

# ------------------------------------------------------------------
# Final save
# ------------------------------------------------------------------
final_df = (pd.DataFrame([
                {"phrase": orig, "pos": "NOUN_PHRASE", "count": c}
                for (_, (orig, c)) in noun_dict.items()
            ])
            .sort_values("count", ascending=False)
            .reset_index(drop=True))

final_df.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Extracted {len(final_df)} unique noun phrases.")
print("🔝 Top 10 most frequent noun phrases:")
print(final_df.head(10)[["phrase", "count"]].to_string(index=False))

🔤 Loading SpaCy model ('en_core_web_sm')...
✅ Using GPU acceleration
📋 Loaded 326 stop words from SpaCy
📂 Loading sentences from: /content/drive/MyDrive/Collab/GhanaNouns/deduped_sentences.csv
📄 Loaded 3963565 sentences
🚀 Starting noun phrase extraction...


Processing:   1%|▏         | 5/397 [01:03<1:21:43, 12.51s/it]

💾 Checkpoint saved (53017 unique noun phrases).


Processing:   3%|▎         | 10/397 [02:05<1:20:56, 12.55s/it]

💾 Checkpoint saved (101026 unique noun phrases).


...

💾 Checkpoint saved (2186220 unique noun phrases).


Processing:  99%|█████████▉| 395/397 [1:29:30<00:32, 16.31s/it]

💾 Checkpoint saved (2210347 unique noun phrases).


Processing: 100%|██████████| 397/397 [1:29:49<00:00, 13.57s/it]



✅ Done! Extracted 2214724 unique noun phrases.
🔝 Top 10 most frequent noun phrases:
     phrase  count
      study 227243
respondents  79305
       data  54631
   research  52838
     people  50895
       time  45405
        use  44719
    results  42060
 researcher  38228
      order  34724


# Replace Noun Phrases with Placeholders

In [ ]:
#!/usr/bin/env python3
"""
Detect noun phrases in each sentence, strip leading qualifiers, and replace the core noun phrase with placeholders like <1>, <2>, etc.
Outputs a deduplicated CSV of sentences with placeholders.
"""

import os
import re
import pandas as pd
from tqdm import tqdm
import spacy

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt"
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_with_placeholders.csv"
BATCH_SIZE = 100
SAVE_EVERY = 5000
# ------------------------------------------------------------------

# -------------------------------
# Define your qualifiers
# -------------------------------
qualifiers = [
    # Articles
    r'^the\s+',
    r'^a\s+',
    r'^an\s+',
    # Demonstratives
    r'^this\s+',
    r'^that\s+',
    r'^these\s+',
    r'^those\s+',
    # Possessive determiners (FIXED: Added word boundaries)
    r'^my\b\s+',
    r'^your\b\s+',
    r'^his\b\s+',
    r'^her\b\s+',
    r'^its\b\s+',
    r'^our\b\s+',
    r'^their\b\s+',
    r'^whose\b\s+',
    # Predeterminers
    r'^all\s+',
    r'^both\s+',
    r'^half\s+',
    # Quantifier determiners
    r'^each\s+',
    r'^every\s+',
    r'^some\s+',
    r'^any\s+',
    r'^no\s+',
    r'^enough\s+',
    r'^many\s+',
    r'^much\s+',
    r'^few\s+',
    r'^several\s+',
    r'^most\s+',
    r'^least\s+',
    # Comparative / limit quantifiers
    r'^more\s+than\s+',
    r'^less\s+than\s+',
    r'^fewer\s+than\s+',
    r'^greater\s+than\s+',
    r'^no\s+more\s+than\s+',
    r'^no\s+less\s+than\s+',
    r'^at\s+least\s+',
    r'^at\s+most\s+',
    r'^up\s+to\s+',
    # Partitive constructions
    r'^many\s+of\s+the\s+',
    r'^much\s+of\s+the\s+',
    r'^few\s+of\s+the\s+',
    r'^some\s+of\s+the\s+',
    r'^most\s+of\s+the\s+',
    r'^all\s+of\s+the\s+',
    r'^none\s+of\s+the\s+',
    r'^any\s+of\s+the\s+',
    r'^each\s+of\s+the\s+',
    # Other common determiners
    r'^another\s+',
    r'^other\s+',
    r'^such\s+',
    r'^certain\s+',
    r'^various\s+',
]

# -------------------------------
# Precompile regex patterns
# -------------------------------
qualifier_patterns = [re.compile(q, flags=re.IGNORECASE) for q in qualifiers]

# -------------------------------
# Function to strip leading qualifiers and find their length
# -------------------------------
def strip_qualifiers_and_get_length(text):
    """
    Strip leading qualifiers from text and return:
    - stripped_text: text without qualifiers
    - qualifier_length: length of the stripped qualifiers
    """
    original_text = text
    qualifier_length = 0
    changed = True

    while changed:
        changed = False
        for pat in qualifier_patterns:
            match = pat.match(text)
            if match:
                qualifier_length += match.end() - match.start()
                text = text[match.end():]  # remove matched qualifier
                changed = True
                break

    stripped_text = text if text.strip() else original_text
    if stripped_text == original_text:
        qualifier_length = 0

    return stripped_text, qualifier_length

# -------------------------------
# Function to check if noun chunk should be replaced
# -------------------------------
def should_replace_noun_chunk(chunk):
    """
    Check if a noun chunk should be replaced with a placeholder.
    Skip pronouns and proper nouns.
    """
    # Skip pronouns (PRON tag) like "you", "I", "he", "she", etc.
    if chunk.root.pos_ == "PRON":
        return False

    # Skip proper nouns (PROPN tag) if desired
    # if chunk.root.pos_ == "PROPN":
    #     return False

    # Skip specific words that shouldn't be replaced
    skip_words = {"you", "i", "me", "we", "us", "he", "him", "she", "her", "it", "they", "them"}
    if chunk.text.lower() in skip_words:
        return False

    return True

# ------------------------------------------------------------------
# Load SpaCy model
# ------------------------------------------------------------------
print("🔤 Loading SpaCy model ('en_core_web_sm')...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# ------------------------------------------------------------------
# Load sentences
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_FILE}")
with open(INPUT_FILE, encoding="utf-8") as f:
    sentences = [ln.strip() for ln in f if ln.strip()]
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Check for existing progress
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    unique_sentences = set(processed_df['sentence'].tolist())
    print(f"🔁 Resuming from {len(unique_sentences)} processed sentences.")
else:
    unique_sentences = set()

# ------------------------------------------------------------------
# Process sentences in batches
# ------------------------------------------------------------------
print("🚀 Starting noun phrase detection and placeholder replacement...")

for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[i:i + BATCH_SIZE]
    docs = list(nlp.pipe(batch, disable=["ner"]))

    for doc in docs:
        original_text = doc.text
        noun_chunks = list(doc.noun_chunks)

        if not noun_chunks:
            modified_sentence = original_text
        else:
            # Sort noun chunks by their start position
            noun_chunks_sorted = sorted(noun_chunks, key=lambda x: x.start_char)
            result_parts = []
            last_end = 0
            placeholder_idx = 0  # Counter for placeholders

            for chunk in noun_chunks_sorted:
                # Add text between last noun chunk and current one
                result_parts.append(original_text[last_end:chunk.start_char])

                # Check if this noun chunk should be replaced
                if should_replace_noun_chunk(chunk):
                    placeholder_idx += 1
                    # Process the noun chunk
                    chunk_text = chunk.text
                    core_chunk, qual_len = strip_qualifiers_and_get_length(chunk_text)

                    if qual_len > 0:
                        # There are qualifiers to keep
                        # Add the qualifiers
                        result_parts.append(chunk_text[:qual_len])

                        # Add space before placeholder if needed
                        if not chunk_text[:qual_len].endswith(' '):
                            result_parts.append(' ')

                        # Add placeholder for the core noun phrase
                        result_parts.append(f"<{placeholder_idx}>")

                        # Ensure space after placeholder if it's not at the end and next character isn't punctuation or space
                        if chunk.end_char < len(original_text):
                            next_char = original_text[chunk.end_char]
                            if next_char not in ' .,;:!?)\'"':
                                result_parts.append(' ')

                        last_end = chunk.end_char
                    else:
                        # No qualifiers found, replace entire chunk
                        # Add space before placeholder if needed
                        if chunk.start_char > 0 and original_text[chunk.start_char - 1] not in ' (':
                            result_parts.append(' ')

                        # Add placeholder
                        result_parts.append(f"<{placeholder_idx}>")

                        # Ensure space after placeholder if it's not at the end
                        if chunk.end_char < len(original_text):
                            next_char = original_text[chunk.end_char]
                            if next_char not in ' .,;:!?)\'"':
                                result_parts.append(' ')

                        last_end = chunk.end_char
                else:
                    # Keep the noun chunk as is (e.g., pronouns)
                    result_parts.append(chunk.text)
                    last_end = chunk.end_char

            # Add remaining text after last noun chunk
            if last_end < len(original_text):
                result_parts.append(original_text[last_end:])

            modified_sentence = ''.join(result_parts)

            # Clean up double spaces that might have been introduced
            modified_sentence = re.sub(r'\s+', ' ', modified_sentence)
            modified_sentence = modified_sentence.strip()

        if modified_sentence.strip():
            unique_sentences.add(modified_sentence)

    # Save checkpoint periodically
    if i > 0 and i % SAVE_EVERY == 0:
        temp_df = pd.DataFrame({'sentence': sorted(list(unique_sentences))})
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Checkpoint saved ({len(unique_sentences)} unique sentences).")

# ------------------------------------------------------------------
# Final save
# ------------------------------------------------------------------
final_df = pd.DataFrame({'sentence': sorted(list(unique_sentences))})
final_df.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Processed {len(sentences)} sentences into {len(final_df)} unique sentences with placeholders.")
print(f"💾 Results saved to: {OUTPUT_FILE}")

# ------------------------------------------------------------------
# Show some examples
# ------------------------------------------------------------------
print("\n📝 Examples of processed sentences:")
print("-" * 80)
for i, sent in enumerate(list(unique_sentences)[:10]):
    print(f"{i+1}. {sent}")
print("-" * 80)

🔤 Loading SpaCy model ('en_core_web_sm')...
📂 Loading sentences from: /content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt
📄 Loaded 1954942 sentences
🚀 Starting noun phrase detection and placeholder replacement...


Processing:   0%|          | 51/19550 [00:24<2:42:49,  2.00it/s]

💾 Checkpoint saved (4919 unique sentences).


Processing:   1%|          | 101/19550 [00:43<1:50:21,  2.94it/s]

💾 Checkpoint saved (9506 unique sentences).


...

💾 Checkpoint saved (1756348 unique sentences).


Processing: 100%|█████████▉| 19501/19550 [2:38:08<05:08,  6.30s/it]

💾 Checkpoint saved (1760572 unique sentences).


Processing: 100%|██████████| 19550/19550 [2:38:23<00:00,  2.06it/s]



✅ Done! Processed 1954942 sentences into 1763936 unique sentences with placeholders.
💾 Results saved to: /content/drive/MyDrive/Collab/GhanaEntities/sentences_with_placeholders.csv

📝 Examples of processed sentences:
--------------------------------------------------------------------------------
1. The <1>, which was attended by the <2> of the <3>, <4>, and <5> from the <6>, marked the <7> of <8> for the <9>.
2. Beyond securing <1>, the <2> emphasised the <3> for a <4> to the <5>.
3. In a <1>, the <2> said it "concluded that the <3> pointed to the <4> of <5> of the <6> contained in the <7>
4. <1> further noted that <2> have changed," from <3> to <4> and so it tells you that the <5> of these <6> involves tracking down their <7>.
5. For us, it is only natural that we leverage trending <1> of <2>.
6. "We stand to gain a <1> if we make it to <2>, from those on the <3> to our <4> in <5>, everybody will have his <6> for making it to the <7>.
7. We are, arguably, the most <1> in <2>,” he sa

# Extract Noun Phrases and Adjective Phrases

In [ ]:
#!/usr/bin/env python3
"""
Extract noun phrases and adjective phrases from each sentence.
Outputs one row per sentence with comma-separated lists of extracted phrases.
Also creates a separate summary file with phrase frequencies.
"""

import os
import pandas as pd
from tqdm import tqdm
import spacy
from collections import Counter

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt"
OUTPUT_SENTENCES = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_with_phrases.csv"
OUTPUT_PHRASE_COUNTS = "/content/drive/MyDrive/Collab/GhanaEntities/phrase_counts.csv"
BATCH_SIZE = 100
SAVE_EVERY = 5000
# ------------------------------------------------------------------

# Load SpaCy model
print("🔤 Loading SpaCy model ('en_core_web_sm')...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

# ------------------------------------------------------------------
# Load sentences
# ------------------------------------------------------------------
print(f"📂 Loading sentences from: {INPUT_FILE}")
with open(INPUT_FILE, encoding="utf-8") as f:
    sentences = [ln.strip() for ln in f if ln.strip()]
print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# Adjective Phrase Extraction Function
# ------------------------------------------------------------------
def extract_adjective_phrases(doc):
    """Extract adjective phrases (e.g., 'very happy', 'not important')."""
    phrases = []
    for token in doc:
        if token.pos_ == "ADJ":
            modifiers = [t for t in token.lefts if t.dep_ in ("advmod", "neg")]
            modifiers.sort(key=lambda t: t.i)
            phrase_tokens = modifiers + [token]
            phrase = " ".join(t.text for t in phrase_tokens)
            if phrase and len(phrase) > 1:
                phrases.append(phrase)
    return phrases

# ------------------------------------------------------------------
# Check for existing progress
# ------------------------------------------------------------------
start_from = 0
sentence_data = []

if os.path.exists(OUTPUT_SENTENCES):
    print(f"🔁 Found existing output: {OUTPUT_SENTENCES}")
    try:
        existing_df = pd.read_csv(OUTPUT_SENTENCES)
        start_from = len(existing_df)
        sentence_data = existing_df.to_dict('records')
        print(f"🔁 Resuming from sentence {start_from}...")
    except Exception as e:
        print(f"⚠️ Could not read existing file: {e}")
        print("🆕 Starting fresh.")
        start_from = 0
        sentence_data = []
else:
    print("🆕 Starting fresh (no existing progress found).")
    sentence_data = []

# ------------------------------------------------------------------
# Process sentences
# ------------------------------------------------------------------
print("🚀 Processing sentences and extracting phrases...")

# Track phrase frequencies across all sentences
noun_counter = Counter()
adj_counter = Counter()

# Process in batches
for i in tqdm(range(start_from, len(sentences), BATCH_SIZE), desc="Processing", initial=start_from, total=len(sentences)):
    batch = sentences[i:i + BATCH_SIZE]
    docs = list(nlp.pipe(batch, disable=["ner"]))

    for idx_in_batch, doc in enumerate(docs):
        sentence = batch[idx_in_batch]

        # Extract noun phrases
        noun_phrases = [chunk.text.strip() for chunk in doc.noun_chunks if chunk.text.strip()]

        # Extract adjective phrases
        adj_phrases = extract_adjective_phrases(doc)

        # Count phrases for frequency analysis
        for phrase in noun_phrases:
            noun_counter[phrase] += 1
        for phrase in adj_phrases:
            adj_counter[phrase] += 1

        # Store sentence data
        sentence_data.append({
            "sentence": sentence,
            "noun_phrases": ", ".join(noun_phrases) if noun_phrases else "",
            "adjective_phrases": ", ".join(adj_phrases) if adj_phrases else "",
            "num_noun_phrases": len(noun_phrases),
            "num_adjective_phrases": len(adj_phrases)
        })

    # Periodic checkpoint save
    if i > 0 and (i % SAVE_EVERY == 0):
        temp_df = pd.DataFrame(sentence_data)
        temp_df.to_csv(OUTPUT_SENTENCES, index=False)
        print(f"💾 Checkpoint saved ({len(sentence_data)} sentences processed).")

# Final save of sentences
print("💾 Saving sentence-level results...")
sentences_df = pd.DataFrame(sentence_data)
sentences_df.to_csv(OUTPUT_SENTENCES, index=False)

# Create phrase frequency summary
print("📊 Creating phrase frequency summary...")

phrase_counts = []
# Add noun phrases
for phrase, count in noun_counter.items():
    phrase_counts.append({"phrase": phrase, "pos": "NOUN_PHRASE", "count": count})
# Add adjective phrases
for phrase, count in adj_counter.items():
    phrase_counts.append({"phrase": phrase, "pos": "ADJ_PHRASE", "count": count})

phrase_counts_df = pd.DataFrame(phrase_counts)
phrase_counts_df = phrase_counts_df.sort_values(["count", "phrase"], ascending=[False, True])
phrase_counts_df.to_csv(OUTPUT_PHRASE_COUNTS, index=False)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
print(f"\n✅ Done! Processed {len(sentences_df)} sentences.")

print(f"\n📊 Sentence-level output saved to: {OUTPUT_SENTENCES}")
print(f"   Columns: sentence, noun_phrases, adjective_phrases, num_noun_phrases, num_adjective_phrases")
print(f"\n📊 Phrase frequency summary saved to: {OUTPUT_PHRASE_COUNTS}")
print(f"   Columns: phrase, pos, count")

print(f"\n🔝 Top 10 most frequent noun phrases:")
noun_df = phrase_counts_df[phrase_counts_df['pos'] == 'NOUN_PHRASE'].head(10)
if not noun_df.empty:
    print(noun_df.to_string(index=False))
else:
    print("   No noun phrases found.")

print(f"\n🔝 Top 10 most frequent adjective phrases:")
adj_df = phrase_counts_df[phrase_counts_df['pos'] == 'ADJ_PHRASE'].head(10)
if not adj_df.empty:
    print(adj_df.to_string(index=False))
else:
    print("   No adjective phrases found.")

# Show some examples from the sentence file
print("\n📋 Sample sentences with extracted phrases:")
sample_df = sentences_df.head(5)
for _, row in sample_df.iterrows():
    print(f"\nSentence: {row['sentence'][:100]}..." if len(row['sentence']) > 100 else f"\nSentence: {row['sentence']}")
    if row['noun_phrases']:
        print(f"  Noun phrases: {row['noun_phrases'][:100]}..." if len(row['noun_phrases']) > 100 else f"  Noun phrases: {row['noun_phrases']}")
    if row['adjective_phrases']:
        print(f"  Adjective phrases: {row['adjective_phrases'][:100]}..." if len(row['adjective_phrases']) > 100 else f"  Adjective phrases: {row['adjective_phrases']}")

🔤 Loading SpaCy model ('en_core_web_sm')...
📂 Loading sentences from: /content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt
📄 Loaded 1954942 sentences
🆕 Starting fresh (no existing progress found).
🚀 Processing sentences and extracting phrases...


Processing:   0%|          | 51/1954942 [00:12<132:59:11,  4.08it/s]

💾 Checkpoint saved (5100 sentences processed).


Processing:   0%|          | 101/1954942 [00:25<135:57:54,  3.99it/s]

💾 Checkpoint saved (10100 sentences processed).


...

Processing:   0%|          | 8902/1954942 [44:29<1327:54:11,  2.46s/it]

💾 Checkpoint saved (890100 sentences processed).


Processing:   0%|          | 8952/1954942 [44:48<1316:52:40,  2.44s/it]

💾 Checkpoint saved (895100 sentences processed).


Processing:   0%|          | 9000/1954942 [45:00<162:12:37,  3.33it/s]

KeyboardInterrupt



# Extract Triples

In [ ]:
#!/usr/bin/env python3
"""
Extract (head, relation, tail) triplets from a **plain-text file**
(one sentence per line) with Babelscape/mrebel-large.
All GPU / batch / OOM / checkpoint tricks kept from the original script.
"""
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt"
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/extracted_entities.csv"
SAVE_EVERY   = 5000
BATCH_SIZE   = 74
LANGUAGE     = "English"   # default if file contains no language column
# ------------------------------------------------------------------

# ------------- MODEL / TOKENIZER identical to original ------------
model_name = "Babelscape/rebel-large"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32     = True

lang_map = {"English": "en_XX", "French": "fr_XX", "Portuguese": "pt_XX"}

# ------------- triplet parser identical to original --------------
def extract_triplets_typed(text: str):
    triplets, current = [], 'x'
    subject = relation = object_ = ''
    text = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").replace("tp_XX", "")
    for token in text.split():
        if token in ("<triplet>", "<relation>"):
            current = 't'
            if subject and relation and object_:
                triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
            subject, relation = '', ''
        elif token.startswith("<") and token.endswith(">"):
            if current in ('t', 'o'):
                current = 's'
                if subject and relation and object_:
                    triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
                object_ = ''
            else:
                current = 'o'
                relation = ''
        else:
            if current == 't':   subject  += ' ' + token
            elif current == 's': object_  += ' ' + token
            elif current == 'o': relation += ' ' + token
    if subject and relation and object_:
        triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
    return triplets

# ------------- batching helpers identical to original ------------
def process_batch(sentences, lang_code):
    tokenizer.src_lang = lang_code
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
        return_attention_mask=True
    )
    inputs = {k: v.to("cuda", non_blocking=True) for k, v in inputs.items()}
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                decoder_start_token_id=tokenizer.convert_tokens_to_ids("tp_XX"),
                max_length=256,
                num_beams=5,
                early_stopping=True,
                do_sample=False,
                num_return_sequences=1
            )
        decoded = tokenizer.batch_decode(outputs.cpu(), skip_special_tokens=False)
        return decoded
    except RuntimeError as e:
        if "out of memory" in str(e):
            return None
        raise

# ------------------------------------------------------------------
# LOAD SENTENCES FROM TXT → create mini DataFrame on the fly
# ------------------------------------------------------------------
with open(INPUT_FILE, encoding="utf-8") as f:
    sentences = [ln.strip() for ln in f if ln.strip()]

df = pd.DataFrame({
    "ID": range(len(sentences)),
    "sentence": sentences,
    "Language": LANGUAGE
})

# ------------------------------------------------------------------
# CHECKPOINT RESUME (same logic as original)
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(processed_df["ID"])
    print(f"🔁 Resuming: {len(processed_df)} rows already processed.")
else:
    processed_df = pd.DataFrame(columns=["ID", "sentence", "head", "relation", "tail"])
    processed_ids = set()

results = []
unprocessed_rows = [row for _, row in df.iterrows() if row["ID"] not in processed_ids]
print(f"📄 {len(sentences)} sentences in file, {len(unprocessed_rows)} left to do.")

# ------------------------------------------------------------------
# MAIN LOOP – almost identical to original, just no language grouping
# ------------------------------------------------------------------
lang_code = lang_map.get(LANGUAGE, "en_XX")

for start in tqdm(range(0, len(unprocessed_rows), BATCH_SIZE), desc="Processing"):
    batch = unprocessed_rows[start:start + BATCH_SIZE]
    sents = [str(r["sentence"]).strip() for r in batch]
    ids   = [r["ID"] for r in batch]

    decoded_texts = process_batch(sents, lang_code)

    if decoded_texts is None:          # OOM → halve batch
        half = len(sents) // 2
        if half > 0:
            for i in range(0, len(sents), half):
                sub_s, sub_i = sents[i:i + half], ids[i:i + half]
                dtx = process_batch(sub_s, lang_code)
                if dtx:
                    for sid, sent, dec in zip(sub_i, sub_s, dtx):
                        for t in extract_triplets_typed(dec):
                            results.append({"ID": sid, "sentence": sent, **t})
    else:
        for sid, sent, dec in zip(ids, sents, decoded_texts):
            for t in extract_triplets_typed(dec):
                results.append({"ID": sid, "sentence": sent, **t})

    # periodic GPU clean-up + save
    if start % (BATCH_SIZE * 10) == 0:
        torch.cuda.empty_cache()
    if len(results) >= SAVE_EVERY:
        temp_df = pd.DataFrame(results)
        processed_df = pd.concat([processed_df, temp_df], ignore_index=True)
        processed_df.to_csv(OUTPUT_FILE, index=False)
        print(f"💾 Saved {len(processed_df)} rows...")
        results.clear()

# final save
if results:
    processed_df = pd.concat([processed_df, pd.DataFrame(results)], ignore_index=True)
processed_df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Done – {len(processed_df)} triplets written to {OUTPUT_FILE}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
import torch

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("✅ GPU cache cleared.")


✅ GPU cache cleared.


In [ ]:
#!/usr/bin/env python3
"""
Extract Named Entities from a plain-text file (one sentence per line)
using Babelscape/wikineural-multilingual-ner.
Outputs a deduplicated list of entities with their types.
"""
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt"
OUTPUT_FILE = "/content/drive/MyDrive/Collab/GhanaEntities/extracted_ner_entities.csv"
SAVE_EVERY = 5000
BATCH_SIZE = 32  # Adjust based on your GPU memory
# ------------------------------------------------------------------

# ------------- MODEL / TOKENIZER SETUP ------------
model_name = "Babelscape/wikineural-multilingual-ner"
print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Compile model for faster inference (PyTorch 2.0+)
try:
    model = torch.compile(model, mode="reduce-overhead")
    print("✅ Model compiled with torch.compile for faster inference")
except Exception as e:
    print(f"⚠️  torch.compile not available: {e}")

# Create NER pipeline with optimized batch processing
nlp = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Use simple instead of deprecated grouped_entities
    batch_size=BATCH_SIZE,  # Enable true batching in the pipeline
    # device=model.device  # Remove device argument when using device_map="auto"
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ------------------------------------------------------------------
# LOAD SENTENCES FROM TXT
# ------------------------------------------------------------------
print(f"Loading sentences from: {INPUT_FILE}")
with open(INPUT_FILE, encoding="utf-8") as f:
    sentences = [ln.strip() for ln in f if ln.strip()]

print(f"📄 Loaded {len(sentences)} sentences")

# ------------------------------------------------------------------
# CHECKPOINT RESUME
# ------------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    processed_count = len(processed_df)
    print(f"🔁 Resuming: {processed_count} entities already found.")
    # Create a set of existing entities for deduplication
    existing_entities = set(
        zip(processed_df["entity"].str.lower(), processed_df["entity_type"])
    )
else:
    processed_df = pd.DataFrame(columns=["entity", "entity_type", "count"])
    existing_entities = set()
    processed_count = 0

# Dictionary to track entity counts: (entity_lower, type) -> (original_form, count)
entity_dict = {}
if not processed_df.empty:
    for _, row in processed_df.iterrows():
        # Skip rows with missing entity or entity_type
        if pd.isna(row["entity"]) or pd.isna(row["entity_type"]):
            continue
        entity_str = str(row["entity"]).strip()
        if not entity_str:  # Skip empty strings
            continue
        key = (entity_str.lower(), row["entity_type"])
        entity_dict[key] = (entity_str, int(row["count"]) if not pd.isna(row["count"]) else 1)

# ------------------------------------------------------------------
# PROCESS SENTENCES IN BATCHES
# ------------------------------------------------------------------
print(f"Starting NER extraction...")

for start in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Processing"):
    batch = sentences[start:start + BATCH_SIZE]
    # Filter out empty sentences
    batch = [s for s in batch if s.strip()]

    if not batch:
        continue

    try:
        # Process entire batch at once - pipeline handles internal batching
        batch_results = nlp(batch, batch_size=BATCH_SIZE)

        # batch_results is a list of lists (one list per sentence)
        for ner_results in batch_results:
            # Extract entities from this sentence
            for entity_info in ner_results:
                entity_text = entity_info["word"].strip()
                entity_type = entity_info["entity_group"]

                # Use lowercase for deduplication key
                key = (entity_text.lower(), entity_type)

                if key in entity_dict:
                    # Increment count
                    orig_form, count = entity_dict[key]
                    entity_dict[key] = (orig_form, count + 1)
                else:
                    # New entity
                    entity_dict[key] = (entity_text, 1)

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"\n⚠️  OOM error at batch {start}. Splitting into smaller chunks...")
            torch.cuda.empty_cache()
            # Split batch in half and retry
            half_size = len(batch) // 2
            if half_size > 0:
                for i in range(0, len(batch), half_size):
                    mini_batch = batch[i:i + half_size]
                    try:
                        mini_results = nlp(mini_batch)
                        for ner_results in mini_results:
                            for entity_info in ner_results:
                                entity_text = entity_info["word"].strip()
                                entity_type = entity_info["entity_group"]
                                key = (entity_text.lower(), entity_type)

                                if key in entity_dict:
                                    orig_form, count = entity_dict[key]
                                    entity_dict[key] = (orig_form, count + 1)
                                else:
                                    entity_dict[key] = (entity_text, 1)
                    except Exception as e2:
                        print(f"\n⚠️  Error on mini-batch: {str(e2)}")
                        continue
        else:
            raise

    # Periodic GPU clean-up and save (much less frequent)
    if start > 0 and start % (BATCH_SIZE * 100) == 0:  # Every 25,600 sentences
        torch.cuda.empty_cache()

        # Save checkpoint
        results = [
            {"entity": orig_form, "entity_type": ent_type, "count": count}
            for (_, ent_type), (orig_form, count) in entity_dict.items()
        ]
        temp_df = pd.DataFrame(results)
        temp_df = temp_df.sort_values(["entity_type", "count"], ascending=[True, False])
        temp_df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n💾 Checkpoint saved: {len(temp_df)} unique entities | Processed {start * BATCH_SIZE:,} sentences")

    # Show progress without saving (every 1000 batches)
    elif start > 0 and start % 1000 == 0:
        print(f"\n📊 Progress: {len(entity_dict)} unique entities found | Processed {start * BATCH_SIZE:,}/{len(sentences):,} sentences")


# ------------------------------------------------------------------
# FINAL SAVE
# ------------------------------------------------------------------
results = [
    {"entity": orig_form, "entity_type": ent_type, "count": count}
    for (_, ent_type), (orig_form, count) in entity_dict.items()
]

final_df = pd.DataFrame(results)
final_df = final_df.sort_values(["entity_type", "count"], ascending=[True, False])
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Done! Extracted {len(final_df)} unique entities")
print(f"📊 Entity type distribution:")
print(final_df["entity_type"].value_counts())
print(f"\n💾 Results saved to: {OUTPUT_FILE}")

# Display sample of most common entities
print("\n🔝 Top 10 most frequent entities:")
print(final_df.nlargest(10, "count")[["entity", "entity_type", "count"]].to_string(index=False))

Loading model: Babelscape/wikineural-multilingual-ner


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
The model 'OptimizedModule' is not supported for ner. Supported models are ['PeftModelForTokenClassification', 'AlbertForTokenClassification', 'ApertusForTokenClassification', 'ArceeForTokenClassification', 'BertForTokenClassification', 'BigBirdForTokenClassification', 'BioGptForTokenClassification', 'BloomForTokenClassification', 'BrosForTokenClassification', 'CamembertForTokenClassification', 'CanineForTokenClassification', 'ConvBertForTokenClassification', 'Data2VecTextForTokenClassification', 'DebertaForTokenClassification', 'DebertaV2ForTokenClassification', 'DeepseekV3ForTokenClassification', 'DiffLlamaForTokenClassification', 'DistilBertForTokenClassification', 'ElectraForTokenClassification', 'ErnieForTokenClassification', 'ErnieMForTokenClassification', 'EsmForTokenClassification', 'Exaone4ForTokenClassification', 'FalconForTokenClassification', 'FlaubertForTokenClassification', 'FNetForTokenClassificat

✅ Model compiled with torch.compile for faster inference
Loading sentences from: /content/drive/MyDrive/Collab/GhanaEntities/sentences_en_clean.txt
📄 Loaded 1954942 sentences
🔁 Resuming: 39592 entities already found.
Starting NER extraction...


Processing:   0%|          | 0/61092 [00:07<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ======== OPTIMIZED SETTINGS =========
INPUT_FILE = "/content/easyblog-articles-export_sentences.csv"
OUTPUT_FILE = "/content/extracted_entities_relations.csv"
SAVE_EVERY = 500
BATCH_SIZE = 74  # Increased batch size with optimizations below

# ======== LOAD MODEL AND TOKENIZER WITH OPTIMIZATIONS =========
model_name = "Babelscape/mrebel-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Optimized model loading
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use mixed precision
    device_map="auto",  # Automatically distribute across available GPUs
    low_cpu_mem_usage=True
)
model.eval()

# Enable better GPU utilization
torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 for matmul
torch.backends.cudnn.allow_tf32 = True  # Allow TF32 for cuDNN

# ======== LANGUAGE MAP =========
lang_map = {
    "English": "en_XX",
    "French": "fr_XX",
    "Portuguese": "pt_XX"
}

# ======== OPTIMIZED FUNCTION: Extract Triplets =========
def extract_triplets_typed(text):
    triplets = []
    relation = ''
    text = text.strip()
    current = 'x'
    subject, relation, object_ = '', '', ''

    # Pre-clean text
    text = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").replace("tp_XX", "")

    for token in text.split():
        if token == "<triplet>" or token == "<relation>":
            current = 't'
            if relation and subject and object_:
                triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token.startswith("<") and token.endswith(">"):
            if current in ['t', 'o']:
                current = 's'
                if relation and subject and object_:
                    triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
                object_ = ''
            else:
                current = 'o'
                relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token

    # Final triplet
    if subject and relation and object_:
        triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})

    return triplets

# ======== BATCH PROCESSING OPTIMIZATIONS =========
def group_by_language(rows):
    """Group rows by language for more efficient batch processing"""
    grouped = {}
    for row in rows:
        lang = row.get("Language", "English")
        if lang not in grouped:
            grouped[lang] = []
        grouped[lang].append(row)
    return grouped

def process_batch(sentences, lang_code):
    """Process a single batch with optimal settings"""
    try:
        tokenizer.src_lang = lang_code

        inputs = tokenizer(
            sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
            return_attention_mask=True
        )

        # Move to GPU with pin_memory if available
        inputs = {k: v.to("cuda", non_blocking=True) for k, v in inputs.items()}

        with torch.no_grad():
            # Use faster generation config
            outputs = model.generate(
                **inputs,
                decoder_start_token_id=tokenizer.convert_tokens_to_ids("tp_XX"),
                max_length=256,
                num_beams=5,  # Balanced between quality and speed
                early_stopping=True,
                do_sample=False,  # Faster than sampling
                num_return_sequences=1
            )

        # Move outputs to CPU for decoding to free GPU memory
        outputs = outputs.cpu()
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)

        return decoded

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"⚠️ OOM with batch size {len(sentences)}, reducing...")
            # Handle OOM by reducing batch size dynamically
            return None
        raise e

# ======== LOAD INPUT DATA =========
df = pd.read_csv(INPUT_FILE)

# ======== LOAD CHECKPOINT IF EXISTS =========
if os.path.exists(OUTPUT_FILE):
    processed_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(zip(processed_df["id"], processed_df["sentence"]))
    print(f"🔁 Resuming from previous run: {len(processed_df)} rows already processed.")
else:
    processed_df = pd.DataFrame(columns=["id", "sentence", "head", "relation", "tail"])
    processed_ids = set()

results = []

# ======== OPTIMIZED MAIN LOOP =========
unprocessed_rows = [row for idx, row in df.iterrows()
                   if (row["id"], row["sentence"]) not in processed_ids]

print(f"📊 Processing {len(unprocessed_rows)} unprocessed rows...")

# Group by language first for more efficient processing
language_groups = group_by_language(unprocessed_rows)

for lang, lang_rows in language_groups.items():
    lang_code = lang_map.get(lang, "en_XX")
    print(f"🔤 Processing {len(lang_rows)} rows in {lang}...")

    # Process in batches by language
    for start in tqdm(range(0, len(lang_rows), BATCH_SIZE), desc=f"Processing {lang}"):
        batch = lang_rows[start:start + BATCH_SIZE]
        sentences = [str(row["sentence"]).strip() for row in batch]
        ids = [row["id"] for row in batch]

        decoded_texts = process_batch(sentences, lang_code)

        if decoded_texts is None:
            # OOM occurred, try with smaller batch size
            half_batch = len(sentences) // 2
            if half_batch > 0:
                for i in range(0, len(sentences), half_batch):
                    sub_batch = sentences[i:i + half_batch]
                    sub_ids = ids[i:i + half_batch]
                    decoded_texts = process_batch(sub_batch, lang_code)
                    if decoded_texts:
                        for sent_id, sent_text, decoded_text in zip(sub_ids, sub_batch, decoded_texts):
                            triplets = extract_triplets_typed(decoded_text)
                            for t in triplets:
                                results.append({
                                    "id": sent_id,
                                    "sentence": sent_text,
                                    "head": t["head"],
                                    "relation": t["type"],
                                    "tail": t["tail"]
                                })

        else:
            # Process successful batch
            for sent_id, sent_text, decoded_text in zip(ids, sentences, decoded_texts):
                triplets = extract_triplets_typed(decoded_text)
                for t in triplets:
                    results.append({
                        "id": sent_id,
                        "sentence": sent_text,
                        "head": t["head"],
                        "relation": t["type"],
                        "tail": t["tail"]
                    })

        # Clear GPU cache periodically
        if start % (BATCH_SIZE * 10) == 0:
            torch.cuda.empty_cache()

        # ====== SAVE PERIODICALLY ======
        if len(results) >= SAVE_EVERY:
            temp_df = pd.DataFrame(results)
            combined = pd.concat([processed_df, temp_df], ignore_index=True)
            combined.to_csv(OUTPUT_FILE, index=False)
            print(f"💾 Saved {len(combined)} total rows so far...")
            processed_df = combined
            processed_ids = set(zip(processed_df["id"], processed_df["sentence"]))
            results = []  # clear buffer

# ======== FINAL SAVE =========
if results:
    temp_df = pd.DataFrame(results)
    combined = pd.concat([processed_df, temp_df], ignore_index=True)
    combined.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Final save complete: {len(combined)} total rows saved.")
else:
    print("✅ No new rows to save.")

print("🎉 Processing complete!")

📊 Processing 5664 unprocessed rows...
🔤 Processing 5664 rows in English...


Processing English:   9%|▉         | 7/77 [00:14<02:18,  1.98s/it]

💾 Saved 522 total rows so far...


Processing English:  18%|█▊        | 14/77 [00:30<02:47,  2.66s/it]

💾 Saved 1045 total rows so far...


...

💾 Saved 4685 total rows so far...


Processing English:  91%|█████████ | 70/77 [02:42<00:15,  2.28s/it]

💾 Saved 5207 total rows so far...


Processing English: 100%|██████████| 77/77 [02:57<00:00,  2.31s/it]

✅ Final save complete: 5694 total rows saved.
🎉 Processing complete!
